## Model Serving

As the class practice, the students will be required to develop local inference server using the `Churn_Modelling_train_test.csv` dataset and MLFlow for online and batch inference.

**About dataset**

This dataset is obained from [kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction?resource=download). It contains information on bank customers who either left the bank or continue to be a customer. The dataset includes the following attributes:

* Customer ID: A unique identifier for each customer
* Surname: The customer's surname or last name
* Credit Score: A numerical value representing the customer's credit score
* Geography: The country where the customer resides (France, Spain or Germany)
* Gender: The customer's gender (Male or Female)
* Age: The customer's age.
* Tenure: The number of years the customer has been with the bank
* Balance: The customer's account balance
* NumOfProducts: The number of bank products the customer uses (e.g., savings account, credit card)
* HasCrCard: Whether the customer has a credit card (1 = yes, 0 = no)
* IsActiveMember: Whether the customer is an active member (1 = yes, 0 = no)
* EstimatedSalary: The estimated salary of the customer
* Exited: Whether the customer has churned (1 = yes, 0 = no)

### Model Training

For this exercice, it is necessary to have a model registered in MLFlow. For this we can, we can use the experiments from session 2.

In [19]:
# import libraries
from pathlib import Path

import pandas as pd
import numpy as np
import joblib
import mlflow
import requests
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

Start the MLflow server with the following command in the terminal: `mlflow server --host 127.0.0.1 --port 8080`.

Now, for the purpose of this exercice, you are required to define again the data transformation logic and save the one hot encoder as a `.pkl` file (if encoder was used during the pipeline).

In [20]:
# Implement transformation logic as in session 2

DROP_COLUMNS = ["RowNumber", "CustomerId", "Surname"]
BINARY_FEATURES = ["Gender"]
ONE_HOT_COLUMNS = ["Geography"]
TARGET = "Exited"


def prepare_features(df):
    df = df.copy()

    y = None
    if TARGET in df.columns:
        y = df[TARGET].copy()
        df = df.drop(columns=[TARGET])

    df = df.drop(columns=DROP_COLUMNS)

    df["Gender"] = df["Gender"].map({
        "Female": 1,
        "Male": 0
    })

    return df, y


def apply_one_hot_encoder(df, encoder):
    df = df.copy()

    encoded_data = encoder.transform(df[ONE_HOT_COLUMNS])
    encoded_columns = encoder.get_feature_names_out(ONE_HOT_COLUMNS)

    encoded_df = pd.DataFrame(
        encoded_data,
        columns=encoded_columns,
        index=df.index
    )

    df = df.drop(columns=ONE_HOT_COLUMNS)
    df = pd.concat([df, encoded_df], axis=1)

    return df

In [21]:
import joblib

if Path("ejercicio 3 test.csv").exists():
    PATH = Path(".")
else:
    PATH = Path("session_3")

df_train_test = pd.read_csv(PATH / "ejercicio 3 test.csv")

X_train_test_raw, y_train_test = prepare_features(df_train_test)

encoder = OneHotEncoder(
    drop="first",
    sparse_output=False,
    handle_unknown="ignore"
)

encoder.fit(X_train_test_raw[ONE_HOT_COLUMNS])

joblib.dump(encoder, PATH / "one_hot_encoder.pkl")

X_train_test = apply_one_hot_encoder(X_train_test_raw, encoder)

print("Encoder saved successfully.")
print("Train/test shape:", X_train_test.shape)

Encoder saved successfully.
Train/test shape: (9001, 12)


In [22]:
# I already have the experiments from session 2, so I will just load them and display the results.

mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

client = MlflowClient()

all_runs = []

for experiment in client.search_experiments():
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

    if not runs.empty:
        runs["experiment_name"] = experiment.name
        all_runs.append(runs)

runs_df = pd.concat(all_runs, ignore_index=True)

runs_df[
    [
        "experiment_name",
        "tags.mlflow.runName",
        "run_id",
        "metrics.accuracy",
        "metrics.f1_score",
        "metrics.precision",
        "metrics.recall"
    ]
].sort_values(by="metrics.accuracy", ascending=False)

,experiment_name,tags.mlflow.runName,run_id,metrics.accuracy,metrics.f1_score,metrics.precision,metrics.recall
0,Keep Practicing - Nicoputs,random-forest-balanced,2db087c0a5754a04b1f98fc9d78bdd15,0.810105,0.598592,0.526860,0.692935
1,Keep Practicing - Nicoputs,decision-tree-deeper,5c8d1213d273425bb0c2c5e41a3056ae,0.744031,0.542205,0.427230,0.741848
2,Practice Experiment - Nicoputs,decision-tree-baseline,fa92e01865e14a9fb4d352285841dd15,0.689617,0.512642,0.377407,0.798913


In [23]:
# Get model URI from selected run

# Select the random forest model from Session 2
selected_run = runs_df[
    runs_df["tags.mlflow.runName"]
    .astype(str)
    .str.contains("random-forest-balanced", case=False, na=False)
].iloc[0]

run_id = selected_run["run_id"]

print("Selected run:", selected_run["tags.mlflow.runName"])
print("Run ID:", run_id)


def find_model_uri(run_id):
    artifacts = client.list_artifacts(run_id)

    for artifact in artifacts:
        if artifact.is_dir:
            files_inside = client.list_artifacts(run_id, artifact.path)

            for file in files_inside:
                if file.path.endswith("MLmodel"):
                    return f"runs:/{run_id}/{artifact.path}"

    raise ValueError("No MLflow model artifact found.")


model_uri = find_model_uri(run_id)

print("Model URI:", model_uri)

Selected run: random-forest-balanced
Run ID: 2db087c0a5754a04b1f98fc9d78bdd15
Model URI: runs:/2db087c0a5754a04b1f98fc9d78bdd15/random-forest-balanced


### Inference

In this part, you are asked to implement a function for batch and online inference methods by providing a model uri. 

In [24]:
# import validation dataset to test inference

validation_file = PATH / "ejercicio 3 val.csv"

if not validation_file.exists():
    validation_file = PATH / "ejercicio 3 validation.csv"

df_validation = pd.read_csv(validation_file)

print("Validation file:", validation_file)
print("Validation shape:", df_validation.shape)

df_validation.head()

Validation file: ejercicio 3 val.csv
Validation shape: (1001, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,8607,15694581,Rawlings,807,Spain,Male,42.0,5,0.00,2,1.0,1.0,74900.90,0
1,4685,15736963,Herring,623,France,Male,43.0,1,0.00,2,1.0,1.0,146379.30,0
2,1732,15721730,Amechi,601,Spain,Female,44.0,4,0.00,2,1.0,0.0,58561.31,0
3,4743,15762134,Liang,506,Germany,Male,59.0,8,119152.10,2,1.0,1.0,170679.74,0
4,4522,15648898,Chuang,560,Spain,Female,27.0,7,124995.98,1,1.0,1.0,114669.79,0


Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [25]:
# transform data — if necessary

encoder = joblib.load(PATH / "one_hot_encoder.pkl")

X_validation_raw, y_validation = prepare_features(df_validation)

X_validation = apply_one_hot_encoder(X_validation_raw, encoder)

print("Validation transformed shape:", X_validation.shape)

X_validation.head()

Validation transformed shape: (1001, 12)


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Geography_nan
0,807,0,42.0,5,0.00,2,1.0,1.0,74900.90,0.0,1.0,0.0
1,623,0,43.0,1,0.00,2,1.0,1.0,146379.30,0.0,0.0,0.0
2,601,1,44.0,4,0.00,2,1.0,0.0,58561.31,0.0,1.0,0.0
3,506,0,59.0,8,119152.10,2,1.0,1.0,170679.74,1.0,0.0,0.0
4,560,1,27.0,7,124995.98,1,1.0,1.0,114669.79,0.0,1.0,0.0


##### Batch Inference

In [26]:
# define a function to implement batch inference with mlflow
def batch_inference(model_uri: str, input: pd.DataFrame):
    model = mlflow.pyfunc.load_model(model_uri)
    predictions = model.predict(input)

    return predictions

In [27]:
# define the model uri that should be used
model_uri = "runs:/2db087c0a5754a04b1f98fc9d78bdd15/random-forest-balanced"

print(model_uri)

runs:/2db087c0a5754a04b1f98fc9d78bdd15/random-forest-balanced


In [28]:
# check the confusion matrix
from sklearn.metrics import confusion_matrix

##### Online Inference

For the online inference, it is required to set up local server. Follow the steps below to configure it:

1. Open a new bash terminal
2. Execute the follwing command `export MLFLOW_TRACKING_URI=http://127.0.0.1:8080` in the terminal. You should specify the port that we are using for MLFlow
3. Execute the following command `mlflow models serve -m runs:/<run_id>/model -p 5000 --no-conda`. Note that `runs:/<run_id>/model` is your model uri.

In [29]:
import requests
import json

In [30]:
# import validation dataset to test inference - just one record

validation_file = PATH / "ejercicio 3 val.csv"

if not validation_file.exists():
    validation_file = PATH / "ejercicio 3 validation.csv"

df_validation_online = pd.read_csv(validation_file).head(1)

df_validation_online

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,8607,15694581,Rawlings,807,Spain,Male,42.0,5,0.0,2,1.0,1.0,74900.9,0


Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [31]:
# transform data - if necessary

encoder = joblib.load(PATH / "one_hot_encoder.pkl")

X_online_raw, y_online = prepare_features(df_validation_online)

X_online = apply_one_hot_encoder(X_online_raw, encoder)

loaded_model_online = mlflow.pyfunc.load_model(model_uri)

model_schema = loaded_model_online.metadata.get_input_schema()
expected_columns = [column.name for column in model_schema.inputs]

X_online = X_online.reindex(columns=expected_columns, fill_value=0)

X_online

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
0,807,42.0,5,0.0,2,1.0,1.0,74900.9,0.0,1.0,0


In [32]:
def get_inference_endpoint(host="http://127.0.0.1", port=5000):
    return f"{host}:{port}/invocations"

url = get_inference_endpoint()

url

'http://127.0.0.1:5000/invocations'

In [33]:
# define a function to implement online inference with mlflow - pandas input
def online_inference_pandas(url: str, input: pd.DataFrame):
    payload = {
        "dataframe_split": input.to_dict(orient="split")
    }

    response = requests.post(
        url,
        json=payload,
        headers={"Content-Type": "application/json"}
    )

    return response

In [34]:
response_pandas = online_inference_pandas(url, X_online)

print("Status code:", response_pandas.status_code)
print("Response:", response_pandas.text)

Status code: 200
Response: {"predictions": [0]}


In [35]:
# define a function to implement online inference with mlflow - json input
def online_inference_json(url: str, input: dict):
    response = requests.post(
        url,
        json=input,
        headers={"Content-Type": "application/json"}
    )

    return response

In [36]:
# define the json as required by MLFlow
input_json = {
    "dataframe_split": X_online.to_dict(orient="split")
}

input_json

{'dataframe_split': {'index': [0],
  'columns': ['CreditScore',
   'Age',
   'Tenure',
   'Balance',
   'NumOfProducts',
   'HasCrCard',
   'IsActiveMember',
   'EstimatedSalary',
   'Geography_Germany',
   'Geography_Spain',
   'Gender_Male'],
  'data': [[807, 42.0, 5, 0.0, 2, 1.0, 1.0, 74900.9, 0.0, 1.0, 0]]}}

In [37]:
response_json = online_inference_json(url, input_json)

print("Status code:", response_json.status_code)
print("Response:", response_json.text)

Status code: 200
Response: {"predictions": [0]}
